In [ ]:
import sklearn
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path
import requests
import tarfile
import io

### Loading Data

In [ ]:
def load_dataset_from_tar(url: str, dataset_name: str) -> pd.Dataframe: 
    # Check if dataset exists
    data_filepath = Path(f"datasets/{dataset_name}")
    if data_filepath.exists(): 
        return pd.read_csv(data_filepath)
    else: 
        Path("datasets").mkdir(parents=True, exist_ok=True)
        response = requests.get(url)
        with tarfile.open(fileobj=io.BytesIO(response.content), mode="r:gz") as tar: 
            f = tar.extractfile("housing/housing.csv")  
            data_filepath.write_bytes(f.read()) 
            return pd.read_csv(data_filepath)

In [ ]:
url = "https://github.com/ageron/data/raw/main/housing.tgz"
dataset_name = "housing_data.csv"
housing_df = load_dataset_from_tar(url, dataset_name)

In [ ]:
housing_df.hist(bins=50, figsize=(12,8))
plt.show()

### Implement Train/Test Split

In [ ]:
print(type(np.random.default_rng()))

In [ ]:
def naive_train_test_split(df: pd.Dataframe, test_ratio: float, seed: float | None) -> tuple[pd.Dataframe]:
    rng = np.random.default_rng(seed=seed)
    shuffled_indicies = rng.permutation(len(df))
    test_size = int(len(df) * test_ratio)
    test_indicies = shuffled_indicies[:test_size]
    train_indicies = shuffled_indicies[test_size:]
    return df.iloc[train_indicies], df.iloc[test_indicies]

train, test = naive_train_test_split(housing_df, .20, 42)
print(train.shape, test.shape)

In [ ]:
from zlib import crc32

def hashed_train_test_split(df: pd.Dataframe, test_ratio: float) -> tuple[pd.Dataframe]:
    df_w_id = df.reset_index()
    
    '''
    CRC32 (Cyclic Redundancy Check) treats your input as a binary polynomial and
    divides it by a fixed 32-bit polynomial, keeping the remainder as the hash.
    Good for fast verification of messages
    '''
    def crc32_hash(id: int) -> int: 
        return crc32(np.int64(id))

    ids = df_w_id["index"]
    test_ids = ids.apply(lambda x: crc32_hash(x) < test_ratio * (2**32))
    return df.iloc[~test_ids], df.iloc[test_ids]

train, test = hashed_train_test_split(housing_df, .20)
print(train.shape, test.shape)

'''
Note: hashed_train_test_split is suitable for retraining model ontop of its current self as it prevents retraining on potentially previous set
- naive method is suitable when you are retraining from scratch as the potential overlap from new and old data is not a concern
'''

In [ ]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(housing_df, test_size=0.2, random_state=42)
print(train.shape, test.shape)

In [ ]:
housing_df["income_cat"] = pd.cut(housing_df["median_income"], bins=[0,1.5,3,4.5,6,np.inf], labels=[1,2,3,4,5])

income_cat_counts = housing_df["income_cat"].value_counts().sort_index()
income_cat_counts.plot.bar(rot=0, grid=True)
plt.show()

from sklearn.model_selection import StratifiedShuffleSplit
splitter = StratifiedShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
splits = []
for train, test in splitter.split(housing_df, housing_df["income_cat"]):
    splits.append([housing_df.iloc[train], housing_df.iloc[test]]) 
train, test = splits[0]
print(train.shape, test.shape)

train, test = train_test_split(housing_df, test_size=0.2, stratify=housing_df["income_cat"], random_state=42)
print(train.shape, test.shape)

### EDA

In [ ]:
train_eda = train.copy()

train_eda.plot(kind="scatter", x="longitude", y="latitude", alpha=0.2)
plt.show()

train_eda.plot(kind="scatter", x="longitude", y="latitude", 
               s=train_eda["population"]/100, label="population", 
               c="median_house_value", cmap="jet", colorbar=True, 
               legend=True, sharex=True, figsize=(10,8))
plt.show()

In [ ]:
'''
Standard correlation coefficient ([-1,1])
'''
corr_matrix = train_eda.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values()

from pandas.plotting import scatter_matrix
attributes = ["median_house_value", "median_income", "total_rooms", "housing_median_age"]
scatter_matrix(train_eda[attributes], figsize=(12, 8))
plt.show()

### Scikit-Learn Design

Note: datasets are represented as `numpy` arrays or `scipy` sparse matrices NOT `pandas` df

**Estimator**: type of base class object that can estimate parameters based on a dataset 
- `fit()`: input = dataset, labels 
- `{estimator}.strategy`: publicly accessible hyperparameters
- `{estimator}.statistics_`: publicly accessible learned params

**Transformers**: type of estimator that can transform a dataset
- `transform()`: input = dataset => output = transformed dataset
- `fit_transform()`: computationally optimized `fit()` + `transform()`
- `inverse_transform()`: returns the inverse of the transformation that the transformer did

**Predictors**: type of estimator that can make predictions
- `predict()`: input = dataset => output = dataset with predictions
- `score()`: measures quality of predictions given test set

### Data Preprocessing

Dealing with `NULL` values in `numpy|pandas`
1. Drop rows (instances)
2. Drop cols 
3. Replace `NULL` values with a substitute (0, mean, median, etc.)

In [ ]:
cleaned_train = train.copy()
cleaned_train.dropna(subset=["total_bedrooms"], inplace=True)

cleaned_train = train.copy()
cleaned_train.drop("total_bedrooms", axis=1, inplace=True)

cleaned_train = train.copy()
median = cleaned_train["total_bedrooms"].median()
cleaned_train["total_bedrooms"] = cleaned_train["total_bedrooms"].fillna(median)
cleaned_train.dropna(subset=["total_bedrooms"], inplace=True)

Dealing with `NULL` values with `sklearn`

**Imputer**: tool that fills in missing values in a dataset 
- `SimpleImputer`: basic mean, median, mode, and constants 
- `KNNImputer`: replaces missing values based on the mean of k-nearest neighbors
- `ImperativeImputer`: trains a regression model for the missing feature based on all other features

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median") # "mean", "most_frequent", "constant" but specify fill_value=...
train_num = train.select_dtypes(include=[np.number])
imputer.fit(train_num)

print(imputer.statistics_)
train_num = imputer.transform(train_num)

Dealing with categorical variables with `sklearn`

**Encoder**: tool that transforms categorical features into numerical features
- `OrdinalEncoder`: transforms categories with ordered numbers
- `OneHotEncoder`: transforms categories into boolean feature of if in category (1) or not (0)

Categories can be found `{encoder}.categories_`

In [ ]:
train_cat = train[["ocean_proximity"]]

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

ordinal_encoder = OrdinalEncoder() # sparse_output=False => output numpy dense array
train_cat_ordinal = ordinal_encoder.fit_transform(train_cat)

onehot_encoder = OneHotEncoder()
train_cat_onehot = onehot_encoder.fit_transform(train_cat)

print(onehot_encoder.feature_names_in_)
print(onehot_encoder.categories_)
print(onehot_encoder.get_feature_names_out())

### Practice Problems

In [97]:
# Create Pipeline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")), 
    ("standardize", StandardScaler()),
])
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")), 
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

num_features = ["longitude", "latitude", "housing_median_age", 
                "total_rooms", "total_bedrooms", "population", 
                "households", "median_income"]
cat_features = ["ocean_proximity"]

from sklearn.compose import ColumnTransformer

preprocessing = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features), 
])

# Process Dataset
X_raw = housing_df.drop("median_house_value", axis=1)
y_raw = housing_df["median_house_value"].values

X_processed = pd.DataFrame(
    preprocessing.fit_transform(X_raw),
    columns=preprocessing.get_feature_names_out()
)
y_processed = housing_df["median_house_value"].values

print(X_processed.columns)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_processed, y_processed, test_size=0.2, random_state=42)

Index(['num__longitude', 'num__latitude', 'num__housing_median_age',
       'num__total_rooms', 'num__total_bedrooms', 'num__population',
       'num__households', 'num__median_income',
       'cat__ocean_proximity_<1H OCEAN', 'cat__ocean_proximity_INLAND',
       'cat__ocean_proximity_ISLAND', 'cat__ocean_proximity_NEAR BAY',
       'cat__ocean_proximity_NEAR OCEAN'],
      dtype='str')


In [ ]:
# Q1, Q2
from sklearn.model_selection import RandomizedSearchCV
from sklearn import svm

small_X_train = X_train[:5000]
small_y_train = y_train[:5000]

params = {
    "kernel": ["rbf", "linear"], 
    "gamma": ["scale", "auto"]
}

random_search = RandomizedSearchCV(svm.SVR(degree=3), params)
random_search.fit(small_X_train, small_y_train)

final_model = random_search.best_estimator_
print("Best Param Set: ", random_search.best_params_)

from sklearn.metrics import root_mean_squared_error

y_hat = final_model.predict(X_test)
final_rmse = root_mean_squared_error(y_test, y_hat)
print("Final RMSE: ", final_rmse)

/Users/kyuminpark/Developer/code/rabbithole/ml/ml_in_sk_pytorch_book/.venv/lib/python3.14/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=10. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


{'kernel': 'linear', 'gamma': 'scale'}
119272.11243227903


In [ ]:
# Q3
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor

feature_filter = Pipeline([
    ("preprocessing", preprocessing), 
    ("filter_features", SelectFromModel(RandomForestRegressor(random_state=42))),
])

feature_filter.fit(X_raw, y_raw)
print(feature_filter.get_feature_names_out())

['num__longitude' 'num__latitude' 'num__median_income'
 'cat__ocean_proximity_INLAND']


In [ ]:
# Q4 
class LatLongKNNTransformer: 
     w